# CropMind AI — Baseline Training on Colab (free GPU route)

This notebook reproduces the **exact Phase 3 experiment** defined by the repository — it does NOT
implement its own training. It runs the repo's own entry point (`ml.training.train`) against the
repo's own config (`ml/configs/train_v1.yaml`): seed 42, MobileNetV3-Small @224px, 21-class
PlantVillage V1 taxonomy, deterministic 70/15/15 split (seed 42), AdamW + cosine + warmup,
early stopping (patience 4). **No hyperparameter, architecture, or dataset changes** vs the repo.

**Cost:** £0 (free T4). **Time:** roughly 45–120 min total.

> **Honesty / provenance policy**
> - Nothing here is a validated metric until published in `reports/model_evaluation/` (Phase 4).
> - The dataset ZIP is uploaded by you to *your own* Drive (PlantVillage CC0 1.0 — keep the folder private, do not re-share). **We never download from Mendeley** (it refuses automated clients, HTTP 403 — documented in `docs/datasets.md`).
> - Optionally pin `EXPECTED_SHA256` from your local `PROVENANCE.json` so this run is provably trained on bytes identical to your verified local import.
> - GPU training is not bit-reproducible between sessions; the `metrics.json` written by *this* run is the canonical artifact behind its checkpoint.

**One-time prerequisites**
1. Create folder `cropmind` in **My Drive**; upload your **existing** `Plant_leaf_diseases_dataset_without_augmentation.zip` into it.
2. Optional but recommended: local `data\raw\plantvillage\PROVENANCE.json` → copy `acquisition.archive_sha256` → paste as `EXPECTED_SHA256` in the config cell.
3. Runtime → Change runtime type → **T4 GPU**.
4. Run cells **top to bottom, one by one**. Training does **not** auto-start: it runs only after you set the confirmation flag and execute the training cell yourself.

## Step 0 — Detect the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — Runtime > Change runtime type > T4 GPU, then re-run."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} ({props.total_memory / (1 << 30):.1f} GB) | torch {torch.__version__}")

## Step 1 — Mount Google Drive + configuration

In [ ]:
from pathlib import Path

# ------------------------------ operator settings ------------------------------
DRIVE_ZIP = Path("/content/drive/MyDrive/cropmind/Plant_leaf_diseases_dataset_without_augmentation.zip")
EXPECTED_SHA256 = ""  # optional: acquisition.archive_sha256 from your local PROVENANCE.json
GIT_URL = "https://github.com/Vishwa-cloud25S/cropmind-ai.git"
COMMIT = "main"  # 'main' = current repository; pin a sha/tag for stronger code reproducibility
CONFIRM_TRAIN = False  # training never auto-runs — set True, then run the training cell explicitly
# -------------------------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")
assert DRIVE_ZIP.exists(), (
    f"Archive not found: {DRIVE_ZIP}. Upload your existing "
    "'Plant_leaf_diseases_dataset_without_augmentation.zip' to My Drive/cropmind/ first."
)

## Step 2 — Verify the archive checksum (never re-download)

When `EXPECTED_SHA256` is populated the notebook **stops with a clear error** on mismatch,
guaranteeing the Colab corpus is byte-identical to the locally verified import.

In [ ]:
import hashlib


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


archive_sha = sha256_file(DRIVE_ZIP)
print("archive sha256:", archive_sha)
if EXPECTED_SHA256:
    assert archive_sha == EXPECTED_SHA256, (
        "CHECKSUM MISMATCH — the Drive archive differs from your verified local import. "
        "Aborting before any processing."
    )
    print("PINNED OK — identical to the sha256 recorded in your local PROVENANCE.json")
else:
    print("WARNING: EXPECTED_SHA256 unset — proceeding unpinned (allowed; pinning is recommended).")

## Step 3 — Clone the repository + install exact dependencies

In [ ]:
# Fresh Colab runtimes are the norm; if a clone already exists (rare), reuse it.
!git clone -q {GIT_URL} /content/cropmind-ai 2>/dev/null || echo "repo already cloned — reusing"
%cd /content/cropmind-ai
!git checkout -q {COMMIT} 2>/dev/null || git checkout -q origin/{COMMIT}
!git --no-pager log --oneline -1
# requirements.txt pins torch>=2.2 / torchvision>=0.17: Colab's pre-installed GPU builds already
# satisfy them, so pip installs only requests/PyYAML/Pillow/numpy/pytest — no CPU wheel pulled.
!python -m pip install -q -r ml/requirements.txt

## Step 4 — Import → split → verify (repository CLI, fail-fast)

Uses the WITHOUT-augmentation tree only (augmentation-leakage policy, `docs/datasets.md`),
writes `PROVENANCE.json`, and reuses the extraction on reruns (idempotent — safe for flaky
Colab sessions). Out-of-scope folders are recorded, never deleted.

In [ ]:
!python -m ml.data.cli import --dataset plantvillage --archive "{DRIVE_ZIP}" --accept-license && \
  python -m ml.data.cli split --dataset plantvillage && \
  python -m ml.data.cli verify --dataset plantvillage

## Step 5 — Dataset statistics + sanity check (computed, not assumed)

The numbers are **calculated** from the fresh split files and then *compared* against the
expected values from the verified local import — they are never hardcoded as the result.
A mismatch stops the notebook before training.

In [ ]:
import json

from ml.data import stats as stats_mod

SPLIT_DIR = Path("data/splits/plantvillage/v1")
summary = stats_mod.generate_stats(SPLIT_DIR, Path("reports/datasets"))  # writes md/csv reports too
manifest = json.loads((SPLIT_DIR / "split_manifest.json").read_text())

computed = {
    "grand_total": summary["grand_total"],
    "class_count": summary["class_count"],
    "train": summary["splits"]["train"],
    "val": summary["splits"]["val"],
    "test": summary["splits"]["test"],
}
expected = {"grand_total": 27335, "class_count": 21, "train": 19126, "val": 4090, "test": 4119}
print("computed:", computed)
print("splits content sha256:", manifest["content_sha256"])
assert computed == expected, (
    f"sanity check failed — computed {computed} vs expected {expected}. "
    "Stop: investigate archive/provenance/split determinism BEFORE training."
)
print("sanity check OK (27,335 images / 21 classes / 19,126 train / 4,090 val / 4,119 test)")

## Step 6 — Train (explicit action required)

**GPU compatibility note:** no hyperparameter changes were needed. MobileNetV3-Small @224x224,
batch 32, peaks at roughly 2–3 GB — far below the T4's ~15 GB. Epochs (12), optimizer (AdamW dual-LR),
scheduler (cosine + warmup), augmentation, and early stopping (patience 4) are all the repo's
unmodified values. `device: auto` selects CUDA automatically; AMP runs on CUDA only; the final
summary prints the recorded device as proof.

**Set `CONFIRM_TRAIN = True` in the Step 1 config cell, then run this cell.** ~598 batches/epoch
(19,126 train images ÷ 32); one summary line prints per epoch. Early stopping may end the run
before epoch 12 — that is the scheduler working, not a failure. Do not interrupt mid-run:
artifacts are written only at completion.

In [ ]:
assert CONFIRM_TRAIN, (
    "Training does not auto-run on notebook open. Set CONFIRM_TRAIN = True in the config cell "
    "(Step 1), then run THIS cell explicitly."
)
!python -m ml.training.train --config ml/configs/train_v1.yaml

## Step 7 — Package + persist the run

The training contract writes `runs/<run_id>/` with `checkpoint.pt` (+ sha256 sidecar),
`metrics.json` (training history, held-out test metrics incl. per-class P/R/F1, dataset content
hash, model metadata), and a `config.yaml` copy. This cell asserts every artifact exists, verifies
the checkpoint sha256, zips the run, copies the ZIP to `My Drive/cropmind/runs/`, and triggers a
browser download.

In [ ]:
import glob
import shutil

run_dir = Path(sorted(glob.glob("runs/*/metrics.json"))[-1]).parent
metrics = json.loads((run_dir / "metrics.json").read_text())

required = ["checkpoint.pt", "checkpoint.sha256", "metrics.json", "config.yaml"]
missing = [name for name in required if not (run_dir / name).exists()]
assert not missing, f"run artifacts missing: {missing}"

ckpt = torch.load(run_dir / "checkpoint.pt", map_location="cpu", weights_only=True)
assert ckpt["meta"]["checkpoint_sha256"] == (run_dir / "checkpoint.sha256").read_text().strip()
print(f"artifacts preserved in {run_dir}: {', '.join(required)} — checkpoint sha256 verified")

bundle = Path(shutil.make_archive(f"/content/{run_dir.name}", "zip", run_dir))
drive_runs = Path("/content/drive/MyDrive/cropmind/runs")
drive_runs.mkdir(parents=True, exist_ok=True)
shutil.copy2(bundle, drive_runs / bundle.name)
print("ZIP copied to Drive:", drive_runs / bundle.name)

from google.colab import files

files.download(bundle)

## Step 8 — Bring the results home (Windows)

1. The `runs-<run_id>.zip` downloaded above (backup in `My Drive/cropmind/runs/`).
2. Extract it into the local repo's `runs\` folder → `runs\<run_id>\checkpoint.pt` + `metrics.json`
   (gitignored by design — datasets and weights never enter git).
3. Paste `metrics.json` to the assistant → the input for **Phase 4** (evaluation report + M1 gate
   measurement).
4. The Colab runtime can be stopped/deleted — ZIP + Drive copy hold everything.
   Nothing in this notebook uploads dataset content to GitHub (data lives only on Colab/Drive).

## Step 9 — Final run summary

In [ ]:
import yaml

cfg = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8"))
print("=== CropMind baseline run — canonical summary (unchanged repo config) ===")
print("run directory       :", run_dir)
print("held-out test top-1 :", round(metrics["test"]["top1"], 4))
print("best validation     :", round(metrics["best_val_top1"], 4), "(top-1)")
print("epochs completed    :", len(metrics["history"]))
print("dataset SHA256      :", metrics["splits_content_sha256"])
print("model version       :", cfg["run"]["model_version"])
print("training device     :", metrics["device"], "(recorded hardware)")